So far we've learnt how to scrape the web, and how to make a request for information from an API. Some websites make APIs even easier. Check out [RapidAPI](https://rapidapi.com/) they take care of writing most of the code for you.

We will use the [AeroDataBox API](https://rapidapi.com/aedbx-aedbx/api/aerodatabox/), which can retrieve all sorts of information about flights and airports. We will show you how to retrieve information about the airports, and then it's up to you to apply this, along with what you've already learnt this week, to **produce a function, which retrieves tomorrows flight information for the major airports in the cities you web scraped**.

In [ ]:
import requests
import pandas as pd

In [ ]:
rapidapi_key = "687292277emsh6620811a3972b04p1a4ee9jsn8c02f9bc139b"

We'll simulate a DataFrame like the kind hosted on our sample database. You should use the table on your own database to find airports for those cities.

In [ ]:
cities_df = pd.DataFrame({
    "city_id": [4, 5, 6],
    "name": ["Berlin", "Paris", "London"],
    "country": ["Germany", "France", "England"],
    "latitude": [52.5200, 48.8567, 51.5072],
    "longitude": [13.4050, 2.3522, -0.1275]
})

cities_df

On the left hand side of the AeroDataBox API page, you'll see a list of options for information that you can retrieve:
> - Flight API
> - Flight Alert API
> - Airport API
> - Aircraft API
> - Industry API
> - Statistical API
> - Miscellaneous API
> - Healthcheck & Status API

1. We want to select `Airport API`

2. Then within Airport API we want to select `Search airports by location`

3. Now in the middle third you'll want to select `Params` and enter the `latitude` and `longitude` of any city to test... we chose Berlin: latitude 52.52 longitude 13.405. The default `radiusKM` of 50km seems fine. And finally set `withFlightInfoOnly` to true, so it will only return airports which have flight data (scheduled or live) available.

4. On the right hand third of the screen you should see a block of code that looks pretty unfamiliar. This is because by default the code is probably set to *(Shell) Curl*. However, we have the power to change this to familiar python. Select the "Target" dropdown box at the top of the code and select `python ` and the "Client" dropdown to `Requests`.

Now you can copy the code to your notebook and it should look a little something like the cell below:

In [ ]:
import requests

url = "https://aerodatabox.p.rapidapi.com/airports/search/location"

querystring = {"lat":"52.52","lon":"13.405","radiusKm":"50","limit":"10","withFlightInfoOnly":"true"}

headers = {
    "x-rapidapi-key": rapidapi_key, # it seems they've actually left this out right now
	"x-rapidapi-host": "aerodatabox.p.rapidapi.com",
	"Content-Type": "application/json"
}

response = requests.get(url, headers=headers, params=querystring)

print(response.json())

We can now turn this into a dataframe using `.json_normalize()`

In [ ]:
pd.json_normalize(response.json()['items'])

It looks like this isn't perfect: Tegel is retired since 2021. Let's keep in mind that we'll need to account for this somehow.

Let's now use this to find the airports around multiple cities

In [ ]:
def get_airports(cities_df):
    # API headers
    headers = {
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }
    url = "https://aerodatabox.p.rapidapi.com/airports/search/location"
    

    # DataFrame to store results
    all_airports = []

    for _, row in cities_df.iterrows():
    # Construct the URL with the latitude and longitude
        querystring = {"lat":row["latitude"],"lon":row["longitude"],"radiusKm":"50","limit":"10","withFlightInfoOnly":"true"}

        # Make the API request
        response = requests.get(url, headers=headers, params=querystring)

        if response.status_code == 200:
            data = response.json()
            airports = pd.json_normalize(data.get('items', []))
            all_airports.append(airports)
        else:
            print(f"WARNING: Failed to retrieve airports for {row["name"]}")

    return pd.concat(all_airports, ignore_index=True)

In [ ]:
get_airports(cities_df)

## **Challenge:** Arrivals information
Using what you have been shown above, plus the skills you've learnt in the last couple of days:
1. In `AeroDataBox API` use the `Flight API` > `FIDS/Schedules: Airport departures and arrivals (by time range)` section
2. Fill out the parameters in the middle third and then copy the `python: requests` code from the right hand third
3. Explore the data you get back. What would be useful in your DataFrame and what can be excluded? Remember Gans wants to know about when people are arriving in the city
4. Make a DataFrame from the information you see as important
5. Condense everything you did above into a function that can take a list of ICAO codes as an input, and as an output gives you a DataFrame with the information for *tomorrows arrivals*

### 
_____

We can start by loading our needed secrets and reading the `'cities'` table from Gans's database.

In [ ]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

In [ ]:
load_dotenv()
connection_string = os.getenv("CON_STRING")
rapidapi_key = os.getenv("RAPIDAPI_KEY")
cities_df = pd.read_sql("cities", con=connection_string)
cities_df

### Step 1: Airports

A function for finding airports is already provided. We'll modify it just a little, selecting only a few columns from each result and renaming one of them.

In [ ]:
def get_airports(cities_df):
    # API constants
    headers = {
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }
    url = "https://aerodatabox.p.rapidapi.com/airports/search/location"

    # DataFrame to store results
    all_airports = []

    for _, row in cities_df.iterrows():
        querystring = {"lat":row["latitude"],"lon":row["longitude"],"radiusKm":"50","limit":"10","withFlightInfoOnly":"true"}
        response = requests.get(url, headers=headers, params=querystring)

        if response.status_code == 200:
            data = response.json()
            airports = pd.json_normalize(data.get('items', []))
            airports["city_id"] = row["city_id"] # add city_id for foreign key reference
            all_airports.append(airports)
        else:
            print(f"WARNING: Failed to retrieve airports for {row["name"]}")

    airports_df = pd.concat(all_airports, ignore_index=True) # make one DataFrame from individual results
    airports_df = airports_df[["icao", "name", "city_id"]] 

    return airports_df

In [ ]:
airports_df = get_airports(cities_df)
airports_df

This result looks just fine. Before proceeding with flights, let's write a table definition to `03_gans_schema.sql` and populate the table with `airports_df`. One thing worth noting, we'll add an extra column for `'active'` to the SQL table. This way if AeroDataBox provides a bad airport (like Tegel) we have a way to exclude it from flight searches. Equally valid, we could work out a method to blacklist certain airports from `airports_df`.

In [ ]:
airports_df.to_sql(
    "airports",
    con=connection_string,
    if_exists="append",
    index=False
)

### Step 2: Flights

Let's begin by reading from the database's `'airports'` table. Here, we select only airports that we've flagged as "active".

In [ ]:
airports_df = pd.read_sql("SELECT * FROM airports WHERE `active` = 1", con=connection_string)
airports_df

#### One airport, one time window

Following the approach taken with weather forecasts, we'll start small with a single call to the API and then add layers of loops to retrieve more and more data.

In [ ]:
# copy headers from above
headers = {
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }

# copy url and querystring from AeroDataBox website, filling in one icao code
url = "https://aerodatabox.p.rapidapi.com/flights/airports/icao/EDDB/2026-08-22T00:00/2026-08-22T11:59"
querystring = {"withLeg":"false","direction":"Arrival","withCancelled":"false","withCodeshared":"false"}

response = requests.get(url, headers=headers, params=querystring)
print(response.json())

We can now begin to explore the response

In [ ]:
response_json = response.json()
response_json.keys()

In [ ]:
# looks like a list
response_json["arrivals"]

In [ ]:
# try to understand one arrival
# check the webpage as well, there may be items not present in every arrival!
arrival = response_json["arrivals"][0]
arrival

In [ ]:
arrival.keys()

In [ ]:
arrival_dict = {
    "depart_airport": arrival["movement"]["airport"]["name"],
    "depart_country": arrival["movement"]["airport"]["countryCode"].upper(), # country codes are more often all capital letters
    "arrive_time_scheduled": arrival["movement"]["scheduledTime"]["local"],
    "arrive_time_revised": arrival["movement"]["revisedTime"]["local"],
    "flight_number": arrival["number"],
    "aircraft": arrival["aircraft"]["model"]
}
arrival_dict

Now let's see if this works with the rest of the flights in the response.

In [ ]:
arrivals = [] # empty list to store arrivals
for arrival in response_json["arrivals"]:
    arrival_dict = {
        "depart_airport": arrival["movement"]["airport"]["name"],
        "depart_country": arrival["movement"]["airport"]["countryCode"].upper(),
        "arrive_time_scheduled": arrival["movement"]["scheduledTime"]["local"],
        "arrive_time_revised": arrival["movement"]["revisedTime"]["local"],
        "flight_number": arrival["number"],
        "aircraft": arrival["aircraft"]["model"]
    }
    arrivals.append(arrival_dict)

flights_df = pd.DataFrame(arrivals)

Looks like we get the same problem we saw with weather: not every key is included in every arrival. `.get()` will help us here again.

In [ ]:
arrivals = [] # empty list to store arrivals
for arrival in response_json["arrivals"]:
    arrival_dict = {
        "depart_airport": arrival["movement"]["airport"]["name"],
        "depart_country": arrival["movement"]["airport"]["countryCode"].upper(),
        "arrive_time_scheduled": arrival["movement"]["scheduledTime"]["local"],
        "arrive_time_revised": arrival["movement"].get("revisedTime", {}).get("local", None), 
        "flight_number": arrival["number"],
        "aircraft": arrival["aircraft"]["model"]
    }
    arrivals.append(arrival_dict)

flights_df = pd.DataFrame(arrivals)
flights_df

Our API will only let us retrieve flight data for up to 12 hours at a time. To cover all of tomorrow, we'll need to build a `for` loop. We'll also need to work out how to get tomorrow's date.

#### One airport, two time windows

In [ ]:
today = pd.Timestamp.now().date()
one_day = pd.Timedelta(1, "day")
tomorrow = today+one_day
tomorrow_str = tomorrow.strftime("%Y-%m-%d")
tomorrow_str

In [ ]:
headers = { # headers stay constant
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }
times = [["00:00", "11:59"], ["12:00", "23:59"]]
airport = airports_df.loc[0] # use the DataFrame this time, instead of copying call details from website

arrivals = [] # empty list to store arrivals
for start_time, end_time in times: # we can "unpack" the interior lists into two iteration variables
    url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{airport["icao"]}/{tomorrow_str}T{start_time}/{tomorrow_str}T{end_time}"
    querystring = {"withLeg":"false","direction":"Arrival","withCancelled":"false","withCodeshared":"false"}
    response = requests.get(url, headers=headers, params=querystring)

    if response.status_code == 200:
        data = response.json()["arrivals"]
        for arrival in data:
            arrival_dict = {
                "arrive_airport": airport["icao"], # add foreign-key information
                "depart_airport": arrival["movement"]["airport"]["name"],
                "depart_country": arrival["movement"]["airport"]["countryCode"].upper(),
                "arrive_time_scheduled": arrival["movement"]["scheduledTime"]["local"],
                "arrive_time_revised": arrival["movement"].get("revisedTime", {}).get("local", None), 
                "flight_number": arrival["number"],
                "aircraft": arrival["aircraft"]["model"]
            }
            arrivals.append(arrival_dict)
        
flights_df = pd.DataFrame(arrivals)
flights_df

Oh no, there's even more keys missing! We'll add some more `.get()` statements.

In [ ]:
headers = { # headers stay constant
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }
times = [["00:00", "11:59"], ["12:00", "23:59"]]
airport = airports_df.loc[0] # use the DataFrame this time, instead of copying call details from website

arrivals = [] # empty list to store arrivals
for start_time, end_time in times: # we can "unpack" the interior lists into two iteration variables
    url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{airport["icao"]}/{tomorrow_str}T{start_time}/{tomorrow_str}T{end_time}"
    querystring = {"withLeg":"false","direction":"Arrival","withCancelled":"false","withCodeshared":"false"}
    response = requests.get(url, headers=headers, params=querystring)

    if response.status_code == 200:
        data = response.json()["arrivals"]
        for arrival in data:
            arrival_dict = {
                "arrive_icao": airport["icao"], # add foreign-key information
                "depart_icao": arrival["movement"]["airport"].get("icao", None), # add this since countryCode and maybe name aren't reliable
                "depart_airport": arrival["movement"]["airport"].get("name", None),
                "depart_country": arrival["movement"]["airport"].get("countryCode", "XX").upper(),
                "arrive_time_scheduled": arrival["movement"]["scheduledTime"]["local"],
                "arrive_time_revised": arrival["movement"].get("revisedTime", {}).get("local", None), 
                "flight_number": arrival.get("number", None),
                "aircraft": arrival.get("aircraft", {}).get("model", None)
            }
            arrivals.append(arrival_dict)
        
flights_df = pd.DataFrame(arrivals)
flights_df

#### All airports, two time windows

In [ ]:
headers = { # headers stay constant
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }
times = [["00:00", "11:59"], ["12:00", "23:59"]]

arrivals = [] # empty list to store arrivals
for _, airport in airports_df.iterrows():
    for start_time, end_time in times: # we can "unpack" the interior lists into two iteration variables
        url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{airport["icao"]}/{tomorrow_str}T{start_time}/{tomorrow_str}T{end_time}"
        querystring = {"withLeg":"false","direction":"Arrival","withCancelled":"false","withCodeshared":"false"}
        response = requests.get(url, headers=headers, params=querystring)
    
        if response.status_code == 200:
            data = response.json()["arrivals"]
            for arrival in data:
                arrival_dict = {
                    "arrive_icao": airport["icao"], # add foreign-key information
                    "depart_icao": arrival["movement"]["airport"].get("icao", None), # add this since countryCode and maybe name aren't reliable
                    "depart_airport": arrival["movement"]["airport"].get("name", None),
                    "depart_country": arrival["movement"]["airport"].get("countryCode", None), # move .upper() to deal with NoneTypes
                    "arrive_time_scheduled": arrival["movement"]["scheduledTime"]["local"],
                    "arrive_time_revised": arrival["movement"].get("revisedTime", {}).get("local", None), 
                    "flight_number": arrival.get("number", None),
                    "aircraft": arrival.get("aircraft", {}).get("model", None)
                }
                arrivals.append(arrival_dict)
        
flights_df = pd.DataFrame(arrivals)
flights_df["depart_country"] = flights_df["depart_country"].str.upper()
flights_df